# Notebook 3 — Loki OT Correction Pipeline

End-to-end demo of the lymphocyte mimicry correction pipeline:
1. Solve the unbalanced optimal transport problem (Loki OT).
2. Obtain hard assignments — mimickers are flagged with label `-1`.
3. Visualise the corrected detections.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from src.loki_ot.cost_matrix import mixed_cost
from src.loki_ot.unbalanced_ot import sinkhorn_kl_unbalanced, compute_transport_assignment
from src.correction.pipeline import CorrectionPipeline

In [ ]:
# ------------------------------------------------------------------
# Quick synthetic demo (no real data needed)
# ------------------------------------------------------------------
np.random.seed(42)

N, K, D = 200, 50, 128   # detected cells, reference cells, feature dim

# Simulated features: lymphocytes cluster near reference; mimickers are spread
source_features  = np.vstack([
    np.random.randn(150, D) * 0.5,          # 150 true lymphocytes
    np.random.randn(50,  D) * 3.0 + 5.0,   # 50 mimickers (far from reference)
]).astype(np.float32)
reference_features = (np.random.randn(K, D) * 0.5).astype(np.float32)

# Tissue labels: all cells in tissue type 0 (stroma)
source_tissue    = np.zeros(N, dtype=np.int64)
reference_tissue = np.zeros(K, dtype=np.int64)

print(f"Source: {source_features.shape} | Reference: {reference_features.shape}")

In [ ]:
# Build cost matrix and solve OT
C = mixed_cost(source_features, reference_features, source_tissue, reference_tissue, alpha=0.7)
print(f"Cost matrix: {C.shape}, range [{C.min():.3f}, {C.max():.3f}]")

a = np.ones(N) / N
b = np.ones(K) / K

T = sinkhorn_kl_unbalanced(a, b, C, epsilon=0.05, rho=1.0, max_iter=500)
print(f"Transport plan shape: {T.shape}")

In [ ]:
# Hard assignment: mimickers get -1
# Use a threshold = mean mass - 1 std to detect outliers
mass_per_cell = T.max(axis=1)
threshold     = mass_per_cell.mean() - mass_per_cell.std()
assignments   = compute_transport_assignment(T, threshold=threshold)

n_lymphocytes = (assignments >= 0).sum()
n_mimickers   = (assignments == -1).sum()
print(f"Kept as lymphocytes: {n_lymphocytes} | Flagged as mimickers: {n_mimickers}")

In [ ]:
# Visualise transport mass distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(mass_per_cell, bins=40, edgecolor="k")
axes[0].axvline(threshold, color="red", linestyle="--", label=f"threshold = {threshold:.4f}")
axes[0].set_xlabel("Max transport mass")
axes[0].set_ylabel("Count")
axes[0].set_title("Transport mass distribution")
axes[0].legend()

colors = ["steelblue" if v >= 0 else "red" for v in assignments]
axes[1].scatter(source_features[:, 0], source_features[:, 1], c=colors, s=10, alpha=0.7)
axes[1].set_title("PCA dim 0-1: blue=lymphocyte, red=mimicker")
axes[1].set_xlabel("Feature dim 0")
axes[1].set_ylabel("Feature dim 1")

plt.tight_layout()
plt.show()